# Phase 1 — Project Setup

**Goal:** connect this repository to a Colab GPU runtime, prepare persistent output folders, and verify PyTorch CUDA operations.

Run cells in order. No dataset is downloaded and no SRGAN training is started in this notebook.

Select **Runtime → Change runtime type → T4 GPU** before starting. GPU availability depends on Colab allocation.


## 1. Get the repository

The code runs in Colab's temporary `/content` storage. Rerunning this cell preserves an existing checkout; it does not discard local changes or automatically update it.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/ennaguiyoussef/SRGAN-Image-Super-Resolution.git"
PROJECT_ROOT = Path("/content/SRGAN-Image-Super-Resolution")

if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", "main", REPO_URL, str(PROJECT_ROOT)], check=True)
elif not (PROJECT_ROOT / ".git").is_dir():
    raise RuntimeError(f"{PROJECT_ROOT} exists but is not a Git checkout.")

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project:", PROJECT_ROOT)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)


## 2. Install supporting dependencies

Keep Colab's preinstalled CUDA-compatible PyTorch and torchvision pair. This requirements file installs supporting packages only. A version snapshot is saved later to record the actual runtime.


In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)


## 3. Verify the GPU

This cell checks CUDA availability and performs an actual GPU matrix multiplication. If it fails, stop here and check the Colab runtime type. The reported GPU may differ from a T4.


In [ ]:
import json
import torch
from src.setup import environment_report, load_config, prepare_directories, seed_everything

report = environment_report()
print(json.dumps(report, indent=2))
if not report["cuda_available"]:
    raise RuntimeError("CUDA is unavailable. Select a GPU runtime and rerun the notebook.")

device = torch.device("cuda:0")
x = torch.randn(128, 128, device=device)
y = x @ x.T
torch.cuda.synchronize()
assert y.shape == (128, 128) and torch.isfinite(y).all().item()
del x, y
print("GPU smoke test passed.")


## 4. Mount Google Drive and prepare folders

Authorize Google Drive when Colab asks. Future training checkpoints, results, and logs will use these persistent folders. Datasets stay in temporary Colab storage for now and must be restored after a runtime reset.

This prepares storage; checkpoint saving and training resume logic will be implemented with the training phase.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/SRGAN-Image-Super-Resolution")
paths = prepare_directories(PROJECT_ROOT, OUTPUT_ROOT)
for name, path in paths.items():
    print(f"{name}: {path}")


## 5. Load configuration and record the environment

The initial training patch sizes are **LR 24 × 24 → HR 96 × 96** at scale ×4. Batch size 16 is a starting value to validate when training is implemented, not a measured memory guarantee.


In [ ]:
config = load_config(PROJECT_ROOT / "configs/srgan.yaml")
seed_everything(config["seed"])
lr_size = config["hr_crop_size"] // config["scale_factor"]
print(json.dumps(config, indent=2))
print(f"Planned patch pair: LR {lr_size}x{lr_size} -> HR {config['hr_crop_size']}x{config['hr_crop_size']}")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
report["git_commit"] = commit
report["config"] = config
(paths["logs"] / "setup_environment.json").write_text(
    json.dumps(report, indent=2), encoding="utf-8"
)
with (paths["logs"] / "requirements-runtime.txt").open("w", encoding="utf-8") as handle:
    subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=handle, check=True)

probe = paths["logs"] / "setup_write_check.txt"
probe.write_text("Storage write check passed.\n", encoding="utf-8")
assert probe.read_text(encoding="utf-8") == "Storage write check passed.\n"
print("Phase 1 complete: GPU, configuration, and persistent storage checks passed.")


## Completion checklist

- GPU smoke test passed.
- Google Drive mounted and storage write check passed.
- Runtime versions and project commit recorded under `logs/` on Drive.

**Next: Phase 2 — select a dataset, split training/validation/test images, and visualize LR/HR pairs.**
